In [ ]:
import gzip
import json
import numpy as np
from monty.json import MontyDecoder
from pymatgen.core import Structure

In [ ]:
!wget https://huggingface.co/datasets/mavrl/matpes/resolve/main/MatPES-R2SCAN-2025.1.json.gz -O ../data/MatPES-R2SCAN-2025.1.json.gz

In [ ]:
MatPes = json.load(gzip.open("../data/MatPES-R2SCAN-2025.1.json.gz", "rt"), cls=MontyDecoder)

In [ ]:
MatPes_4_sites = [el for el in MatPes if el["nsites"] == 4]

In [ ]:
#set random generator seed for reproducibility
np.random.seed(42)
random_ids = np.random.choice(len(MatPes_4_sites), 1000, replace=False)
random_selection_4_sites = [MatPes_4_sites[i] for i in random_ids]


In [23]:
from estimate_cost import get_cost_info_from_structure
# Extract cost information from random_selection_4_sites
cost_info = [
    get_cost_info_from_structure(
        el["composition"],
        el["structure"].lattice.matrix,
        el["symmetry"]["number"]
    )
    for el in random_selection_4_sites
]

nbands, kpts, spgs, nkpts_irreducible, costs = zip(*cost_info)
mat_pes_ids = [el["matpes_id"] for el in MatPes_4_sites]

# Filter some small materials for quick validation
selected_indices = [
    i for i, (n, nk) in enumerate(zip(nbands, nkpts_irreducible))
    if n <= 24 and nk <= 200
]

nbands = [nbands[i] for i in selected_indices]
kpts = [kpts[i] for i in selected_indices]
spgs = [spgs[i] for i in selected_indices]
nkpts_irreducible = [nkpts_irreducible[i] for i in selected_indices]
costs = [costs[i] for i in selected_indices]
mat_pes_ids = [mat_pes_ids[i] for i in selected_indices]
batch = [random_selection_4_sites[i] for i in selected_indices]

print(f"Number of materials selected: {len(batch)}")

# Set magmom for each selected structure
for el in batch:
    el["magmom"] = el["structure"].site_properties.get("magmom", [])

Number of materials selected: 139


In [ ]:
# we will add all this to the database makes a comparison easier
metadata_keys = ["provenance", 'symmetry', 'energy', 'forces', 'stress', 'bandgap', 'matpes_id', 'functional', 'formation_energy_per_atom', 'cohesive_energy_per_atom', "magmom"]

In [ ]:
batch_metadata_general = {"Description": "MATPES validation structures with 4 sites","BANDS24": 1, "Phase": "test"}

In [26]:
import run_utilities
import run_calculation

batch_metadata = batch_metadata_general.copy()
batch_metadata.update({
    'min_nkpts_irreducible': np.min(nkpts_irreducible),
    'max_nkpts_irreducible': np.max(nkpts_irreducible),
    'min_cost': np.min(costs),
    'max_cost': np.max(costs),
    'min_nbands': np.min(nbands),
    'max_nbands': np.max(nbands)
})

structure_list = [el["structure"] for el in batch]

# renaming 'matpes_id' to 'mat_id'
metadata_list = [
    {
        (key if key != "matpes_id" else "mat_id"): calculation[key]
        for key in metadata_keys
    }
    for calculation in batch
]

In [ ]:
run_utilities.run_batch(
    metadata_list=metadata_list,
    structure_list=structure_list,
    batch_metadata=batch_metadata,
    calc_func=run_calculation.static_calculation,
    ncore=2,
    kpar=1,
    nbands=nbands,
    nkpts=nkpts_irreducible,
    ignore_memory_check=True,
    worker="euler_2", # Adjust this worker name as needed
    launchpad_path="/home/sjonathan/atomate/config/lematrho_launchpad.yaml",  # Adjust this path as needed
)